# Forge Neo for Google Colab

Stable Diffusion WebUI **Forge Neo専用**の起動ノートブックです。既存のForge本体・設定・拡張機能には触れません。

- NVIDIA A100を前提とし、既定ではA100以外のGPUで停止します。
- Checkpoint、LoRA、VAE、EmbeddingはGoogle Drive上の既存資産を参照します。
- Animaのモデルは「3-2」のセルで配布元から自動でGoogle Driveへ保存できます（2回目以降はスキップ）。
- Forge Neo本体、Python環境、拡張機能、Florence-2キャッシュ、設定、生成画像はランタイム終了時に消えます。
- 生成画像は `/content/forge-neo-outputs` のみに保存されます。必要な画像はランタイムを終了する前にZIPでダウンロードしてください。
- Civitaiから取得したモデル、プレビュー、メタデータはGoogle Driveに残ります。モデルごとのライセンスと利用条件を確認してください。
- 共有URLは認証付きです。パスワードはセッションごとに生成され、起動セルで一度表示されます。

この版の固定コミットは2026-09-02に取得可能であることとソース構成を確認した初期固定値です。A100上の生成受入テストは、使用モデルを指定して実行する段階で完了させてください。

**目的:** 構成と注意事項の確認　**所要時間:** 1分

## 1. ユーザー設定

通常は変更せず実行できます。最新版検証モードは再現性がなくなるため、検証時だけ有効にしてください。

**目的:** 固定バージョン、拡張機能、パス、認証、APIを一か所で設定　**所要時間:** 数秒

In [ ]:
from pathlib import Path
from collections import deque
import json, os, queue, re, secrets, shutil, subprocess, sys, textwrap, threading, time, uuid

# Forge Neo本体
FORGE_REPOSITORY = "https://github.com/Haoming02/sd-webui-forge-classic"
FORGE_BRANCH = "neo"
FORGE_NEO_SHA = "0d0cb72951b059c8ea17861ba86db8d0f6098c28"
USE_LATEST_FORGE_NEO = False
CLEAN_REBUILD = True  # 毎回Forge本体とvenvを作り直し、失敗した前回状態を持ち越さない
REQUIRE_A100 = True  # FalseならA100以外でも警告後に続行（初版の動作保証外）

# 全拡張機能を初期状態でインストール。Falseにすると個別に除外できます。
EXTENSIONS = {
    "adetailer-neo": {"enabled": True, "required": True, "repo": "https://github.com/Haoming02/ADetailer-Neo", "sha": "67e973e1c5019c16dcf838364ae391a2752bfa7b"},
    "sd-forge-couple": {"enabled": True, "required": True, "repo": "https://github.com/Haoming02/sd-forge-couple", "sha": "c7884e81623d7fcbf4c92e3aea14ced8b5b6aa74"},
    "sd-webui-florence-2-neo": {"enabled": True, "required": True, "repo": "https://github.com/AMGarkin/sd-webui-florence-2-neo", "sha": "96e1b7942f85b362cad6d85a723a64ae471a1e49"},
    "sd-webui-civitai-downloader": {"enabled": True, "required": True, "repo": "https://github.com/otacoo/sd-webui-civitai-downloader", "sha": "2de1348f89d3a60974d3007a4eee63af40a682f2"},
}

MODEL_ROOT = Path("/content/drive/MyDrive/StableDiffusion/models")
FORGE_DIR = Path("/content/sd-webui-forge-neo")
VENV_DIR = FORGE_DIR / "venv"
OUTPUT_ROOT = Path("/content/forge-neo-outputs")
CACHE_ROOT = Path("/content/forge-neo-cache")
ZIP_PATH = Path("/content/forge-neo-download.zip")
WEBUI_LOG_PATH = Path("/content/forge-neo-webui.log")
PYTHON_VERSION = "3.13.12"  # 固定Forge NeoコミットのREADME推奨値
# ColabではCUDA 13用ドライバが保証されないため、Forge公式Wikiの旧PyTorch構成を既定にします。
USE_COLAB_COMPATIBLE_TORCH = True
COLAB_TORCH_VERSION = "2.10.0+cu126"
COLAB_TORCHVISION_VERSION = "0.25.0+cu126"
COLAB_TORCH_INDEX = "https://download.pytorch.org/whl/cu126"
UPSTREAM_TORCH_VERSION = "2.13.0+cu130"
UPSTREAM_TORCHVISION_VERSION = "0.28.0+cu130"
UPSTREAM_TORCH_INDEX = "https://download.pytorch.org/whl/cu130"
CIVITAI_SECRET_NAME = "CIVITAI_API_TOKEN"
WEBUI_USERNAME = "forge"
ENABLE_API = False
# Forge本体・組み込みControlNet・ADetailerを同じvenvで共存させるための制約。
# onnx 1.23.0以降はprotobuf>=6.31.1を要求し、Forge固定のprotobuf==4.25.9と両立しません。
PIP_COMPATIBILITY_PINS = {
    "protobuf": "4.25.9",
    "onnx": "1.22.0",
    "onnxruntime": "1.23.2",
    "mediapipe": "0.10.35",
    "ultralytics": "8.3.253",
}
PIP_CONSTRAINT_PATH = CACHE_ROOT / "forge-neo-constraints.txt"
NOTEBOOK_REVISION = "2026-10-17-anima-download-v8"
CREATE_MISSING_MODEL_DIRS = True
CONFIRM_DELETE_OUTPUTS = False

MODEL_LINK_DIRS = ["Stable-diffusion", "Lora", "VAE", "embeddings", "ControlNet", "ControlNetPreprocessor", "ESRGAN", "text_encoder", "diffusion_models"]
# 固定Forgeコミットに追跡されている案内用ファイル。Git内の原本と照合した場合だけ除去可能。
FORGE_MODEL_PLACEHOLDER_PATHS = {
    "models/Stable-diffusion/Put Checkpoint here.txt",
    "models/Lora/Put LoRA here.txt",
    "models/VAE/Put VAE here.txt",
    "models/embeddings/Put Textual Inversion here.txt",
    "models/ESRGAN/Put Upscaler here.txt",
    "models/text_encoder/Put Text Encoders here.txt",
}
MODEL_EXTENSIONS = {".safetensors", ".ckpt", ".pt", ".pth", ".bin", ".gguf"}

def run_checked(args, *, cwd=None, env=None, capture=False, label="処理", fatal_markers=(), startup_fatal_markers=(), log_path=None, url_credentials=None):
    """子プロセスの出力をColabへ逐次表示し、失敗時は末尾ログを例外にも残す。"""
    command = [str(x) for x in args]
    print(f"▶ {label}", flush=True)
    process = subprocess.Popen(command, cwd=cwd, env=env, text=True,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               bufsize=1, errors="replace")
    lines = [] if capture else deque(maxlen=240)
    output_queue, fatal_marker = queue.Queue(), None
    public_url_seen = False
    assert process.stdout is not None
    def read_output():
        log_file = open(log_path, "w", encoding="utf-8") if log_path is not None else None
        try:
            for line in process.stdout:
                if log_file is not None:
                    log_file.write(line)
                    log_file.flush()
                output_queue.put(line)
        finally:
            if log_file is not None:
                log_file.close()
            output_queue.put(None)
    reader = threading.Thread(target=read_output, daemon=True)
    reader.start()
    started = time.monotonic()
    try:
        while True:
            try:
                line = output_queue.get(timeout=30)
            except queue.Empty:
                if process.poll() is None and not capture:
                    elapsed = int(time.monotonic() - started)
                    print(f"… {label}を実行中です（{elapsed}秒経過）", flush=True)
                continue
            if line is None:
                break
            lines.append(line)
            if not capture:
                print(line, end="", flush=True)
                if url_credentials is not None and "Running on public URL:" in line:
                    username, password = url_credentials
                    print(f"  ユーザー名: {username}", flush=True)
                    print(f"  パスワード: {password}", flush=True)
            if fatal_marker is None:
                active_markers = tuple(fatal_markers) + (() if public_url_seen else tuple(startup_fatal_markers))
                fatal_marker = next((marker for marker in active_markers if marker in line), None)
            if "Running on public URL:" in line:
                public_url_seen = True
            # Forgeは拡張読込例外を握りつぶして続行するため、traceback末尾まで取得して停止する。
            if fatal_marker is not None and line.strip() == "---" and process.poll() is None:
                process.terminate()
        returncode = process.wait()
    except KeyboardInterrupt:
        print(f"\n{label}を停止しています...", flush=True)
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    output = "".join(lines)
    result = subprocess.CompletedProcess(command, returncode, stdout=output, stderr="")
    if fatal_marker is not None:
        tail = "".join(lines[-120:]).strip()
        raise RuntimeError(f"{label}のログに致命的パターン {fatal_marker!r} を検出したため停止しました。\n--- ログ末尾 ---\n{tail}")
    if returncode != 0:
        tail = "".join(lines[-120:]).strip() or "子プロセスから出力がありませんでした。"
        raise RuntimeError(f"{label}に失敗しました（終了コード: {returncode}）。\n--- ログ末尾 ---\n{tail}")
    return result

def git_head(path):
    return run_checked(["git", "-C", path, "rev-parse", "HEAD"], capture=True, label=f"{Path(path).name}のコミット確認").stdout.strip()

def count_models(path):
    return sum(1 for p in Path(path).rglob("*") if p.is_file() and p.suffix.lower() in MODEL_EXTENSIONS)

def audit_pip_check(python, env, *, label):
    """pipの宣言上の不整合を確認し、固定Forge由来の既知の1件だけを許容する。"""
    result = subprocess.run([str(python), "-m", "pip", "check"], env=env,
                            text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=False)
    lines = [] if result.returncode == 0 else [line.strip() for line in result.stdout.splitlines() if line.strip()]
    unexpected = []
    for line in lines:
        known = line.startswith("gradio 4.40.0 has requirement pillow") and "but you have pillow 12.3.0" in line
        if known:
            print("⚠️ 上流の既知メタデータ競合を許容:", line)
        else:
            unexpected.append(line)
    if unexpected:
        raise RuntimeError(f"{label}で未許容の依存関係競合があります。\n" + "\n".join(unexpected))
    if not lines:
        print(f"✅ {label}: 依存関係競合なし")

print("設定を読み込みました。拡張機能はすべて初期状態で有効です。")

## 2. GPU・ランタイム診断

GPU、VRAM、ドライバ、Python、ディスク容量を確認します。A100以外で続行するには、設定セルの `REQUIRE_A100` を明示的に `False` にしてください。

**目的:** Colabランタイムが初版の前提を満たすか確認　**所要時間:** 数秒

In [ ]:
print(f"システムPython: {sys.version.split()[0]}")
disk = shutil.disk_usage("/content")
print(f"/content 空き容量: {disk.free / 1024**3:.1f} GiB / {disk.total / 1024**3:.1f} GiB")
if disk.free < 35 * 1024**3:
    raise RuntimeError("ディスク空き容量が35 GiB未満です。PyTorchとForgeの構築途中で不足する可能性があります。ランタイムを再作成してください。")
gpu = run_checked(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader,nounits"],
                  capture=True, label="NVIDIA GPU診断").stdout.strip().splitlines()
if not gpu:
    raise RuntimeError("GPUを認識できません。Colabのランタイム設定でGPUを選択してください。")
print("GPU名 / VRAM(MiB) / Driver:")
for line in gpu:
    print("  " + line)
is_a100 = any("A100" in line.upper() for line in gpu)
try:
    nvidia_driver_major = int(gpu[0].rsplit(",", 1)[1].strip().split(".")[0])
except Exception as exc:
    raise RuntimeError(f"NVIDIAドライバのバージョンを解析できません: {gpu[0]}") from exc
if nvidia_driver_major < 580:
    print("ℹ️ ドライバ580未満のためCUDA 13ランタイムは利用できません。Colab互換のCUDA 12.6構成を使用します。")
elif USE_COLAB_COMPATIBLE_TORCH:
    print("ℹ️ CUDA 13対応ドライバですが、再現性を優先してColab互換のCUDA 12.6構成を使用します。")
if not is_a100:
    message = "A100ではありません。初版の動作保証対象外です。"
    if REQUIRE_A100:
        raise RuntimeError(message + " 続行する場合はREQUIRE_A100=Falseに変更してください。")
    print("⚠️ " + message + " 設定により続行します。")
else:
    print("✅ A100を認識しました。")

## 3. Google Driveのマウントとモデルフォルダ確認

既存モデルは移動・上書き・削除しません。不足フォルダは空のフォルダだけ新規作成します。書き込み確認用の一時ファイルは直後に削除します。

**目的:** 共通モデル資産とCivitai保存先を準備　**所要時間:** 1～2分（Google認証を含む）

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

drive_root = Path("/content/drive/MyDrive").resolve()
model_root_resolved = MODEL_ROOT.resolve(strict=False)
if not model_root_resolved.is_relative_to(drive_root):
    raise RuntimeError(f"モデルルートがGoogle Drive配下ではありません: {MODEL_ROOT}")
if not MODEL_ROOT.exists():
    if CREATE_MISSING_MODEL_DIRS:
        MODEL_ROOT.mkdir(parents=True, exist_ok=True)
        print(f"モデルルートを新規作成しました: {MODEL_ROOT}")
    else:
        raise RuntimeError(f"共通モデルルートがありません: {MODEL_ROOT}")

for name in MODEL_LINK_DIRS:
    target = MODEL_ROOT / name
    if not target.exists():
        if CREATE_MISSING_MODEL_DIRS:
            target.mkdir(parents=False, exist_ok=False)
            print(f"空フォルダを作成: {target}")
        else:
            raise RuntimeError(f"必要なモデルフォルダがありません: {target}")
    probe = target / f".forge_neo_write_test_{uuid.uuid4().hex}"
    try:
        probe.write_text("write-test", encoding="utf-8")
    except Exception as exc:
        raise RuntimeError(f"書き込み権限がありません: {target} ({exc})") from exc
    finally:
        probe.unlink(missing_ok=True)

checkpoint_count = count_models(MODEL_ROOT / "Stable-diffusion")
print(f"Checkpoint候補: {checkpoint_count}件")
if checkpoint_count == 0:
    print("⚠️ Checkpointがありません。起動前検証までにGoogle Driveへ配置するかCivitaiから取得してください。")
for vae_name in ["vae-ft-mse-840000-ema-pruned.safetensors", "sdxl_vae.safetensors"]:
    print(("✅" if (MODEL_ROOT / "VAE" / vae_name).exists() else "⚠️"), vae_name)

## 3-2. Animaモデルの自動ダウンロード

配布元（Hugging Face の circlestone-labs/Anima）から、Animaの本体・テキストエンコーダー・VAEをGoogle Driveへ保存します。すでに同じファイルがある場合はダウンロードしません。自分で用意したモデルだけを使う場合は `DOWNLOAD_ANIMA` のチェックを外してください。

- 本体 → `StableDiffusion/models/Stable-diffusion`
- テキストエンコーダー（qwen_3_06b_base）→ `StableDiffusion/models/text_encoder`
- VAE（qwen_image_vae）→ `StableDiffusion/models/VAE`

Animaは **CircleStone Labs Non-Commercial License**（非商用ライセンス）です。ダウンロードする前に https://huggingface.co/circlestone-labs/Anima/blob/main/LICENSE.md を確認してください。

**目的:** Animaをすぐ使える状態にする　**所要時間:** 初回5～10分（合計約5.6GB）、2回目以降は数秒

In [ ]:
DOWNLOAD_ANIMA = True  # @param {type:"boolean"}
ANIMA_MODEL = "anima-aesthetic-v1.1"  # @param ["anima-aesthetic-v1.1", "anima-turbo-v1.1", "anima-base-v1.0"]

import requests

ANIMA_REPO = "circlestone-labs/Anima"
ANIMA_REVISION = "f973fc41ec7545364ac9776c2440285f43ff2a30"  # 2026-10-17に確認した版
anima_files = [
    (f"split_files/diffusion_models/{ANIMA_MODEL}.safetensors", MODEL_ROOT / "Stable-diffusion" / f"{ANIMA_MODEL}.safetensors"),
    ("split_files/text_encoders/qwen_3_06b_base.safetensors", MODEL_ROOT / "text_encoder" / "qwen_3_06b_base.safetensors"),
    ("split_files/vae/qwen_image_vae.safetensors", MODEL_ROOT / "VAE" / "qwen_image_vae.safetensors"),
]

def download_hf_file(repo_path, target):
    url = f"https://huggingface.co/{ANIMA_REPO}/resolve/{ANIMA_REVISION}/{repo_path}"
    if target.exists() and target.stat().st_size > 0:
        print(f"⏭️ 取得済み: {target.name}")
        return
    partial = target.with_name(target.name + ".part")
    partial.unlink(missing_ok=True)
    with requests.get(url, stream=True, timeout=60) as response:
        response.raise_for_status()
        total = int(response.headers.get("content-length", 0))
        done, next_report = 0, 0
        with partial.open("wb") as handle:
            for chunk in response.iter_content(chunk_size=16 * 1024 * 1024):
                handle.write(chunk)
                done += len(chunk)
                if total and done >= next_report:
                    print(f"  {target.name}: {done / 1024**3:.2f} / {total / 1024**3:.2f} GiB", flush=True)
                    next_report += total // 10
    if total and partial.stat().st_size != total:
        partial.unlink(missing_ok=True)
        raise RuntimeError(f"{target.name} のダウンロードが途中で切れました。もう一度このセルを実行してください。")
    partial.rename(target)
    print(f"✅ 保存しました: {target}")

if DOWNLOAD_ANIMA:
    for repo_path, target in anima_files:
        download_hf_file(repo_path, target)
    print("Forge Neoの画面で、Checkpointに", f"{ANIMA_MODEL}.safetensors", "を選んでください。")
else:
    print("Animaの自動ダウンロードをスキップしました。")

## 4. Forge Neo本体と専用Python環境の構築

既定では前回のForge本体とvenvをランタイム内から除去し、`uv`でPython 3.13.12の専用環境を毎回クリーンに作成して固定コミットへdetached checkoutします。Google Drive上のモデルと `/content/forge-neo-outputs` は削除対象になりません。PyTorchはForge任せにせず、Colabでのドライバ互換性を考慮した構成を先に導入・検証します。最新版モードでもGoogle Drive上のモデルは更新対象になりません。

既定の `USE_COLAB_COMPATIBLE_TORCH=True` は、Forge公式Wikiの旧PyTorch手順にある PyTorch 2.10.0 + CUDA 12.6を使用します。固定Forgeコミット既定のCUDA 13.0はNVIDIAドライバ580以上が必要なので、Colabでは互換構成を優先します。

Colabが設定することのある `UV_SYSTEM_PYTHON` は明示的に除去し、`VIRTUAL_ENV`、`UV_PYTHON`、`PATH`をForge専用環境へ固定します。PyTorchは `uv --python <venvのpython>` で導入しますが、Forgeの残り依存関係と拡張機能は専用Pythonの `python -m pip` を使い、`--uv` フックによる `/usr` への誤インストール経路を構造的に避けます。

Forge内蔵拡張の無指定依存が将来の最新版へ変動しないよう、すべての子 `pip` に共通の制約ファイルを適用します。ONNXはForge固定の `protobuf==4.25.9` と両立する最終版の `onnx==1.22.0` に固定します。

**目的:** Forge NeoとPyTorch/CUDA依存関係を再現可能な状態で構築　**所要時間:** 初回10～25分

In [ ]:
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
# Colab/コンテナ向けのシステムPython強制設定を、このノートブックの子プロセスへ引き継がない。
removed_uv_system = os.environ.pop("UV_SYSTEM_PYTHON", None)
if removed_uv_system is not None:
    print(f"ℹ️ UV_SYSTEM_PYTHON={removed_uv_system!r} を解除しました（Forge専用venvを保護）。")
removed_transformers_cache = os.environ.pop("TRANSFORMERS_CACHE", None)
if removed_transformers_cache is not None:
    print("ℹ️ 非推奨のTRANSFORMERS_CACHEを現在のColabプロセスから解除しました。")
os.environ.update({
    "HF_HOME": str(CACHE_ROOT / "huggingface"),
    "HUGGINGFACE_HUB_CACHE": str(CACHE_ROOT / "huggingface" / "hub"),
    "TORCH_HOME": str(CACHE_ROOT / "torch"),
    "XDG_CACHE_HOME": str(CACHE_ROOT / "xdg"),
    "UV_CACHE_DIR": str(CACHE_ROOT / "uv"),
    "PIP_CACHE_DIR": str(CACHE_ROOT / "pip"),
    "PIP_DEFAULT_TIMEOUT": "120",
    "UV_HTTP_TIMEOUT": "120",
    "UV_HTTP_RETRIES": "5",
    "GRADIO_ANALYTICS_ENABLED": "False",
    "MPLBACKEND": "Agg",  # Colabのinline backendをheadless Web UIへ引き継がない
    "TF_ENABLE_ONEDNN_OPTS": "0",
    "USE_TORCH": "1",
    "USE_FLAX": "0",
    "USE_TF": "0",
    "GIT_TERMINAL_PROMPT": "0",
    "PIP_DISABLE_PIP_VERSION_CHECK": "1",
})
for proxy_bypass_name in ("NO_PROXY", "no_proxy"):
    proxy_bypass = [value.strip() for value in os.environ.get(proxy_bypass_name, "").split(",") if value.strip()]
    for local_host in ("127.0.0.1", "localhost"):
        if local_host not in proxy_bypass:
            proxy_bypass.append(local_host)
    os.environ[proxy_bypass_name] = ",".join(proxy_bypass)

expected_forge_dir = Path("/content/sd-webui-forge-neo")
if FORGE_DIR != expected_forge_dir:
    raise RuntimeError(f"Forge削除保護: 固定パスと一致しません: {FORGE_DIR}")
if CLEAN_REBUILD and FORGE_DIR.exists():
    if FORGE_DIR.is_symlink() or FORGE_DIR.resolve() != expected_forge_dir:
        raise RuntimeError(f"Forge削除保護: 実体パスが不正です: {FORGE_DIR}")
    # Driveへのディレクトリリンクを明示的に解除してから、ランタイム側だけを削除する。
    clean_models_dir = FORGE_DIR / "models"
    for name in MODEL_LINK_DIRS:
        clean_link = clean_models_dir / name
        if clean_link.is_symlink():
            clean_link.unlink()
            print(f"クリーンアップ前にモデルリンクだけを解除: {clean_link}")
    shutil.rmtree(FORGE_DIR)
    print(f"前回のForge本体とvenvをクリーンアップしました: {FORGE_DIR}")
base_uv_env = os.environ.copy()
base_uv_env.pop("UV_SYSTEM_PYTHON", None)
run_checked([sys.executable, "-m", "pip", "install", "-q", "uv"], env=base_uv_env, label="uvの導入")
run_checked(["uv", "python", "install", PYTHON_VERSION], env=base_uv_env, label=f"Python {PYTHON_VERSION}の準備")

if not FORGE_DIR.exists():
    run_checked(["git", "clone", "--filter=blob:none", "--branch", FORGE_BRANCH, FORGE_REPOSITORY, FORGE_DIR], label="Forge Neoのclone")
elif not (FORGE_DIR / ".git").is_dir():
    raise RuntimeError(f"{FORGE_DIR} が存在しますがGitリポジトリではありません。新しいランタイムでやり直してください。")

remote = run_checked(["git", "-C", FORGE_DIR, "remote", "get-url", "origin"], capture=True, label="Forge Neo取得元の確認").stdout.strip().removesuffix(".git")
if remote != FORGE_REPOSITORY.removesuffix(".git"):
    raise RuntimeError(f"Forge Neoの取得元が想定と異なります: {remote}")
# CLEAN_REBUILD=Falseでセル6まで実行済みの場合、Driveの実体には触れずリンク自体だけを解除する。
# これによりGit checkoutがモデルリンク配下を扱う可能性をなくす。
existing_models_dir = FORGE_DIR / "models"
for name in MODEL_LINK_DIRS:
    existing_link = existing_models_dir / name
    if existing_link.is_symlink():
        existing_link.unlink()
        print(f"再構築前にモデルリンクだけを解除: {existing_link}")
if USE_LATEST_FORGE_NEO:
    print("⚠️ 最新版検証モードです。動作確認済み固定コミットではありません。")
    run_checked(["git", "-C", FORGE_DIR, "fetch", "--depth", "1", "origin", FORGE_BRANCH], label="neoブランチ最新版の取得")
    run_checked(["git", "-C", FORGE_DIR, "checkout", "--detach", "FETCH_HEAD"], label="最新版へのcheckout")
else:
    run_checked(["git", "-C", FORGE_DIR, "fetch", "--depth", "1", "origin", FORGE_NEO_SHA], label="固定コミットの取得")
    run_checked(["git", "-C", FORGE_DIR, "checkout", "--detach", FORGE_NEO_SHA], label="固定コミットへのcheckout")

venv_python = VENV_DIR / "bin" / "python"
if not venv_python.exists():
    run_checked(["uv", "venv", VENV_DIR, "--python", PYTHON_VERSION, "--seed"], env=base_uv_env, label="専用仮想環境の作成")
version = run_checked([venv_python, "-c", "import sys; print('.'.join(map(str, sys.version_info[:3])))"], capture=True, label="専用Pythonの確認").stdout.strip()
if not version.startswith("3.13."):
    raise RuntimeError(f"専用環境のPythonが3.13系ではありません: {version}。新しいランタイムで再構築してください。")

venv_env = os.environ.copy()
venv_env.pop("UV_SYSTEM_PYTHON", None)
venv_env.pop("TRANSFORMERS_CACHE", None)
venv_env.update({
    "VIRTUAL_ENV": str(VENV_DIR),
    "UV_PYTHON": str(venv_python),
    "PATH": str(VENV_DIR / "bin") + os.pathsep + venv_env.get("PATH", ""),
    "UV_LINK_MODE": "hardlink",
    "MPLBACKEND": "Agg",
})

if USE_COLAB_COMPATIBLE_TORCH:
    torch_version, torchvision_version, torch_index = COLAB_TORCH_VERSION, COLAB_TORCHVISION_VERSION, COLAB_TORCH_INDEX
else:
    if nvidia_driver_major < 580:
        raise RuntimeError("CUDA 13構成にはNVIDIAドライバ580以上が必要です。USE_COLAB_COMPATIBLE_TORCH=Trueへ戻してください。")
    torch_version, torchvision_version, torch_index = UPSTREAM_TORCH_VERSION, UPSTREAM_TORCHVISION_VERSION, UPSTREAM_TORCH_INDEX
print(f"PyTorch構成: torch={torch_version}, torchvision={torchvision_version}, index={torch_index}")

# PyTorchを独立した工程として導入し、依存関係全体の失敗と混同しないようにします。
run_checked(["uv", "pip", "install", "--python", venv_python,
             f"torch=={torch_version}", f"torchvision=={torchvision_version}",
             "--index", torch_index], env=venv_env, label="Colab互換PyTorchの導入")
torch_probe_code = "import torch; print('PyTorch:',torch.__version__); print('Built CUDA:',torch.version.cuda); print('CUDA available:',torch.cuda.is_available()); print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'なし')"
torch_probe = run_checked([venv_python, "-c", torch_probe_code], capture=True, label="PyTorch・CUDA事前検証").stdout.strip()
print(torch_probe)
if "CUDA available: True" not in torch_probe:
    raise RuntimeError("PyTorchの導入には成功しましたがCUDAを初期化できません。新しいGPUランタイムで再実行してください。上のDriverとBuilt CUDAも確認してください。")

# Forgeと各拡張が内部で個別に実行するpipにも同じ制約を適用する。
# 制約ファイルはパッケージを勝手に導入せず、導入要求があった場合の版だけを固定します。
PIP_CONSTRAINT_PATH.write_text("".join(f"{name}=={wanted}\n" for name, wanted in PIP_COMPATIBILITY_PINS.items()), encoding="utf-8")
venv_env["PIP_CONSTRAINT"] = str(PIP_CONSTRAINT_PATH)
print(f"pip互換制約: {PIP_CONSTRAINT_PATH}")
for name, wanted in PIP_COMPATIBILITY_PINS.items():
    print(f"  {name}=={wanted}")

# Forgeが最初に必要とするpackagingを、専用Pythonのpipで明示導入する。
run_checked([venv_python, "-m", "pip", "install", "packaging==26.2"], env=venv_env, label="venvへのpackaging導入")
packaging_location = run_checked([venv_python, "-c", "import packaging,sys; print(packaging.__file__); print(sys.prefix)"], capture=True, label="packaging導入先の確認").stdout.strip()
print(packaging_location)
if str(VENV_DIR) not in packaging_location:
    raise RuntimeError(f"packagingがForge専用venvへ入りませんでした。UV_SYSTEM_PYTHONを確認してください。\n{packaging_location}")

env = venv_env.copy()
env.update({
    "WEBUI_LAUNCH_LIVE_OUTPUT": "1",
    "UV_LINK_MODE": "hardlink",
    "TORCH_INDEX_URL": torch_index,
    "TORCH_COMMAND": f"pip install torch=={torch_version} torchvision=={torchvision_version} --index-url {torch_index}",
    "PYTORCH_VERSION": torch_version,
})
run_checked([venv_python, FORGE_DIR / "launch.py", "--skip-version-check", "--exit"], cwd=FORGE_DIR, env=env, label="Forge Neo残り依存関係の導入（venv pip固定）")
dependency_probe = run_checked([venv_python, "-c", "import packaging,gradio,torch; print('packaging:',packaging.__version__); print('gradio:',gradio.__version__); print('torch:',torch.__version__)"], capture=True, label="Forge依存関係の最終確認").stdout.strip()
print(dependency_probe)

# 固定ForgeはPillow 12.3を要求しますが、固定Gradio 4.40のメタデータはPillow<11のままです。
# この上流由来の既知競合だけを許容し、それ以外のpip checkエラーは停止します。
audit_pip_check(venv_python, venv_env, label="Forge本体導入後のpip check")

# 宣言上の競合ではなく、実際にGradioとPillow 12が同時利用できることを確認します。
smoke_code = "from io import BytesIO; import gradio as gr; from PIL import Image; b=BytesIO(); Image.new('RGB',(8,8),(1,2,3)).save(b,format='PNG'); assert len(b.getvalue())>0; c=gr.Image(); print('Gradio/Pillow smoke test: OK')"
smoke_result = run_checked([venv_python, "-c", smoke_code], capture=True, label="Gradio・Pillow実動作確認").stdout.strip()
print(smoke_result)
disk_after = shutil.disk_usage("/content")
print(f"構築後の空き容量: {disk_after.free / 1024**3:.1f} GiB")
if disk_after.free < 8 * 1024**3:
    print("⚠️ 空き容量が8 GiB未満です。大きなモデルやFlorence-2の取得前にランタイム容量を確認してください。")
print(f"✅ Forge Neo: {git_head(FORGE_DIR)} / Python: {version} / torch: {torch_version}")

## 5. 標準拡張機能の導入

全拡張機能を独立して固定コミットへ配置し、各 `install.py` を検証します。有効な必須拡張が1つでも失敗した場合は停止します。ADetailer-NeoとForge CoupleはUIへ読み込みますが、生成機能の既定値はOFFです。

**目的:** ADetailer-Neo、Forge Couple、Florence-2 Neo、Civitai Downloaderを導入　**所要時間:** 3～15分

In [ ]:
extensions_dir = FORGE_DIR / "extensions"
extensions_dir.mkdir(parents=True, exist_ok=True)
extension_failures = []
for name, spec in EXTENSIONS.items():
    if not spec["enabled"]:
        print(f"⏭️ 無効: {name}")
        continue
    path = extensions_dir / name
    try:
        if not path.exists():
            run_checked(["git", "clone", "--filter=blob:none", spec["repo"], path], label=f"{name}のclone")
        elif not (path / ".git").is_dir():
            raise RuntimeError(f"{path} はGitリポジトリではありません")
        remote = run_checked(["git", "-C", path, "remote", "get-url", "origin"], capture=True, label=f"{name}取得元の確認").stdout.strip().removesuffix(".git")
        if remote != spec["repo"].removesuffix(".git"):
            raise RuntimeError(f"取得元が想定と異なります: {remote}")
        run_checked(["git", "-C", path, "fetch", "--depth", "1", "origin", spec["sha"]], label=f"{name}固定コミットの取得")
        # 固定SHAを指すローカルブランチにする。detached HEADはForgeの拡張情報読取りで例外になる。
        run_checked(["git", "-C", path, "checkout", "-B", "codex-pinned", spec["sha"]], label=f"{name}固定コミットへのcheckout")
        installer = path / "install.py"
        if installer.exists():
            install_env = venv_env.copy()
            install_env.pop("UV_SYSTEM_PYTHON", None)
            install_env["PYTHONPATH"] = str(FORGE_DIR) + os.pathsep + install_env.get("PYTHONPATH", "")
            install_env.pop("COMMANDLINE_ARGS", None)  # installerはvenv Pythonのpipを直接使用
            run_checked([venv_python, installer], cwd=FORGE_DIR, env=install_env, label=f"{name}の依存関係導入")
        print(f"✅ {name}: {git_head(path)}")
    except Exception as exc:
        print(f"❌ {name}: {exc}")
        if spec["required"]:
            extension_failures.append(name)
if extension_failures:
    raise RuntimeError("必須拡張機能の導入に失敗しました: " + ", ".join(extension_failures) + "。ログを確認し、新しいランタイムで再構築してください。")

# 組み込みControlNetとADetailerの主要依存が、存在するだけでなく互換固定値どおりか確認する。
distribution_json = run_checked([venv_python, "-c", "import importlib.metadata as m,json; print(json.dumps({d.metadata['Name']: d.version for d in m.distributions()}))"], env=venv_env, capture=True, label="venvパッケージ一覧の取得").stdout.strip()
installed_distributions = {re.sub(r"[-_.]+", "-", name).lower(): version for name, version in json.loads(distribution_json).items()}
required_extension_packages = {name: PIP_COMPATIBILITY_PINS[name] for name in ("onnx", "onnxruntime", "mediapipe")}
if EXTENSIONS["adetailer-neo"]["enabled"]:
    required_extension_packages["ultralytics"] = PIP_COMPATIBILITY_PINS["ultralytics"]
extension_dependency_failures = []
for name, wanted in required_extension_packages.items():
    actual = installed_distributions.get(name)
    print(f"{name}: {actual or '未導入'}（期待値 {wanted}）")
    if actual != wanted:
        extension_dependency_failures.append(f"{name}: {actual or '未導入'}（期待値 {wanted}）")
if extension_dependency_failures:
    raise RuntimeError("拡張機能の互換固定依存が不一致です:\n" + "\n".join(extension_dependency_failures))
print("✅ 拡張機能の互換固定依存を確認しました。")
# Forgeの組み込み・追加拡張は、それぞれのinstall.pyから依存パッケージを導入します。
# その解決過程でprotobufなどForge本体の==固定値が新しい版へ更新されることがあるため、
# 全拡張の導入後に本体requirements.txtを再適用し、再現可能な状態へ戻します。
run_checked([venv_python, "-m", "pip", "install", "-r", FORGE_DIR / "requirements.txt"],
            cwd=FORGE_DIR, env=venv_env, label="拡張機能導入後のForge固定依存再適用")

# pip checkだけではrequirements.txtの==固定値からの逸脱を検出できないため、ここでも直接照合する。
distribution_json = run_checked([venv_python, "-c", "import importlib.metadata as m,json; print(json.dumps({d.metadata['Name']: d.version for d in m.distributions()}))"], env=venv_env, capture=True, label="固定依存再適用後のパッケージ一覧取得").stdout.strip()
installed_distributions = {re.sub(r"[-_.]+", "-", name).lower(): version for name, version in json.loads(distribution_json).items()}
repin_failures = []
for raw in (FORGE_DIR / "requirements.txt").read_text(encoding="utf-8").splitlines():
    declaration = raw.split(";", 1)[0].strip()
    if not declaration or declaration.startswith("#") or "==" not in declaration:
        continue
    package, wanted = (part.strip() for part in declaration.split("==", 1))
    actual = installed_distributions.get(re.sub(r"[-_.]+", "-", package).lower())
    if actual != wanted:
        repin_failures.append(f"{package}: {actual or '未導入'}（期待値 {wanted}）")
for package, wanted in required_extension_packages.items():
    actual = installed_distributions.get(re.sub(r"[-_.]+", "-", package).lower())
    if actual != wanted:
        repin_failures.append(f"{package}: {actual or '未導入'}（互換期待値 {wanted}）")
if repin_failures:
    raise RuntimeError("拡張機能導入後にForge固定依存を復元できませんでした:\n" + "\n".join(repin_failures))
print("✅ 拡張機能導入後のForge固定依存を復元しました。")
onnx_import = run_checked([venv_python, "-c", "import google.protobuf,onnx,onnxruntime; from onnx import TensorProto,helper; node=helper.make_node('Constant',[],['x'],value=helper.make_tensor('v',TensorProto.FLOAT,[1],[1.0])); graph=helper.make_graph([node],'smoke',[],[helper.make_tensor_value_info('x',TensorProto.FLOAT,[1])]); model=helper.make_model(graph); onnx.checker.check_model(model); print('protobuf:',google.protobuf.__version__); print('ONNX:',onnx.__version__); print('ONNX Runtime:',onnxruntime.__version__); print('ONNX/Protobuf smoke test: OK')"], env=venv_env, capture=True, label="ONNX・protobuf再固定後の実動作診断").stdout.strip()
print(onnx_import)
if EXTENSIONS["adetailer-neo"]["enabled"]:
    adetailer_import = run_checked([venv_python, "-c", "import mediapipe,ultralytics; print('MediaPipe import: OK'); print('Ultralytics import: OK')"], env=venv_env, capture=True, label="固定依存再適用後のADetailer import診断").stdout.strip()
    print(adetailer_import)
audit_pip_check(venv_python, venv_env, label="拡張機能導入後のpip check")

## 6. モデル参照・出力・初期設定

Civitai Downloaderが相対パス `models/...` を使い、Florence-2 Neoが `models/LLM` を使うため、`--model-ref` ではなく仕様書のフォールバック（フォルダ単位のシンボリックリンク）を採用します。共有対象だけDriveへリンクし、`models/LLM` はランタイム内に残します。リンク先の既存ファイルは変更しません。

Forge本体に同梱された `Put ... here.txt` は、現在checkout中のGitコミットに追跡された原本と内容が一致する場合だけランタイム側から除去してリンクへ置き換えます。OSによるCRLF/LF変換だけは正規化して比較します。未知のファイル、フォルダ、モデルが存在する場合は安全のため停止します。

CivitaiトークンはColab Secretsの `CIVITAI_API_TOKEN` から任意で読み込み、ランタイム内の設定だけに保存します。Civitaiの削除ボタンは既存モデル保護のため非表示にします。

**目的:** Drive参照、一時出力、Automatic VAE、キャッシュ、拡張無効化状態を設定　**所要時間:** 数秒

In [ ]:
local_models = FORGE_DIR / "models"
local_models.mkdir(parents=True, exist_ok=True)
for name in MODEL_LINK_DIRS:
    link = local_models / name
    target = MODEL_ROOT / name
    if link.is_symlink():
        if link.resolve() != target.resolve():
            raise RuntimeError(f"既存リンクの参照先が異なります: {link} -> {link.resolve()}")
        continue
    if link.exists():
        if not link.is_dir():
            raise RuntimeError(f"ディレクトリではない既存パスをリンクへ置換しません: {link}")
        entries = list(link.iterdir())
        verified_placeholders, unexpected_entries = [], []
        for entry in entries:
            relative_path = entry.relative_to(FORGE_DIR).as_posix()
            if relative_path in FORGE_MODEL_PLACEHOLDER_PATHS and entry.is_file() and not entry.is_symlink():
                try:
                    committed_bytes = subprocess.check_output(
                        ["git", "-C", str(FORGE_DIR), "show", f"HEAD:{relative_path}"],
                        stderr=subprocess.STDOUT,
                    )
                except subprocess.CalledProcessError:
                    committed_bytes = None
                # Git/OSによる改行変換だけを許容し、それ以外の内容変更は許容しない。
                normalize_newlines = lambda data: data.replace(b"\r\n", b"\n")
                if committed_bytes is not None and normalize_newlines(entry.read_bytes()) == normalize_newlines(committed_bytes):
                    verified_placeholders.append(entry)
                    continue
            unexpected_entries.append(entry)
        if unexpected_entries:
            details = "\n".join(f"  - {p} ({'directory' if p.is_dir() else 'file'})" for p in unexpected_entries)
            raise RuntimeError("ユーザーファイルの可能性があるため、既存パスをリンクへ置換しません:\n" + details)
        for placeholder in verified_placeholders:
            placeholder.unlink()  # 固定コミットと内容が一致したランタイム内の案内ファイルだけ
            print(f"案内ファイルを確認して除去: {placeholder}")
        link.rmdir()  # 空になったForgeランタイム側ディレクトリだけを対象
    link.symlink_to(target, target_is_directory=True)

# Florence-2のLLMモデルはリンクしないため、ここ（ランタイム）へ保存されます。
(local_models / "LLM").mkdir(parents=True, exist_ok=True)
for subdir in ["txt2img", "img2img", "extras", "grids", "saved", "temp"]:
    (OUTPUT_ROOT / subdir).mkdir(parents=True, exist_ok=True)
# config.jsonのtemp_dirはGradioコンポーネントのUI構築中から使用されるため、起動前に実体を作る。
webui_temp_dir = CACHE_ROOT / "webui-temp"
webui_temp_dir.mkdir(parents=True, exist_ok=True)

civitai_token = ""
try:
    from google.colab import userdata
    civitai_token = userdata.get(CIVITAI_SECRET_NAME) or ""
except Exception:
    pass

disabled_extensions = [name for name, spec in EXTENSIONS.items() if not spec["enabled"]]
config_path = FORGE_DIR / "config.json"
config = {}
if config_path.exists():
    try:
        config = json.loads(config_path.read_text(encoding="utf-8"))
    except Exception as exc:
        raise RuntimeError(f"既存のランタイム設定を読めません: {config_path} ({exc})") from exc
config.update({
    "VERSION_UID": "PY313",  # 固定Forgeコミットのmodules.launch_utils.VERSION_UIDと一致
    "outdir_txt2img_samples": str(OUTPUT_ROOT / "txt2img"),
    "outdir_hires_samples": str(OUTPUT_ROOT / "txt2img"),
    "outdir_img2img_samples": str(OUTPUT_ROOT / "img2img"),
    "outdir_extras_samples": str(OUTPUT_ROOT / "extras"),
    "outdir_grids": str(OUTPUT_ROOT / "grids"),
    "outdir_save": str(OUTPUT_ROOT / "saved"),
    "outdir_init_images": str(OUTPUT_ROOT / "img2img" / "init-images"),
    "temp_dir": str(CACHE_ROOT / "webui-temp"),
    "sd_vae": "Automatic",
    "disabled_extensions": disabled_extensions,
    "disable_all_extensions": "none",
    "civitai_api_key": civitai_token,
    "civitai_preferred_domain": "civitai.com",
    "civitai_card_button_delete": False,
    "civitai_enable_folder_selector": False,
})
config_path.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")
os.chmod(config_path, 0o600)
print("✅ 出力先、モデルリンク、VAE=Automaticを設定しました。")
print("Civitai APIトークン:", "Secretsから読み込み済み" if civitai_token else "未登録（公開モデルのみ利用可能な場合があります）")
print("ADetailer-Neo / Forge Couple: UIに読み込み、生成時の既定値はOFF")

## 7. 起動前検証

GPU、PyTorch、CUDA、固定コミット、モデル、VAE、出力先、シンボリックリンク、起動引数を検証します。Checkpointがない場合は起動しません。

**目的:** 不整合をWeb UI起動前に日本語で検出　**所要時間:** 1～5分（モデル件数により変動）

In [ ]:
diagnostic = run_checked([venv_python, "-c", "import torch,sys; print('Python:',sys.version.split()[0]); print('PyTorch:',torch.__version__); print('CUDA:',torch.version.cuda); print('CUDA available:',torch.cuda.is_available()); print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'なし'); print('VRAM GiB:',round(torch.cuda.get_device_properties(0).total_memory/1024**3,1) if torch.cuda.is_available() else 0)"],
                         capture=True, label="Python・PyTorch・CUDA診断").stdout.strip()
print(diagnostic)
if "CUDA available: True" not in diagnostic:
    raise RuntimeError("PyTorchがCUDAを利用できません。ランタイムを再作成し、GPU設定と上のインストールログを確認してください。")
if REQUIRE_A100 and "A100" not in diagnostic.upper():
    raise RuntimeError("PyTorchからA100を確認できません。GPUランタイムを再確認してください。")

# Matplotlibだけを独立検査する。ControlNetはForge固有のinitialize_forge()後でなければ正しくimportできないため、
# ここで直接importせず、通常起動ログの『Error loading script』を起動セルで検出する。
matplotlib_env = venv_env.copy()
matplotlib_env["MPLBACKEND"] = "Agg"
matplotlib_probe = run_checked([venv_python, "-c", "import os,matplotlib; print('MPLBACKEND:',os.environ.get('MPLBACKEND')); print('Matplotlib backend:',matplotlib.get_backend())"],
                               env=matplotlib_env, capture=True, label="Matplotlibバックエンド診断").stdout.strip()
print(matplotlib_probe)
if "MPLBACKEND: Agg" not in matplotlib_probe or "Matplotlib backend: Agg" not in matplotlib_probe:
    raise RuntimeError("MatplotlibのheadlessバックエンドをAggへ固定できませんでした。")

# Forgeのrequirements.txtにある単純な == 固定指定を、取得済みメタデータと直接照合する。
distribution_json = run_checked([venv_python, "-c", "import importlib.metadata as m,json; print(json.dumps({d.metadata['Name']: d.version for d in m.distributions()}))"], env=venv_env, capture=True, label="venvパッケージ一覧の再取得").stdout.strip()
installed_distributions = {re.sub(r"[-_.]+", "-", name).lower(): version for name, version in json.loads(distribution_json).items()}
pinned_failures = []
for raw in (FORGE_DIR / "requirements.txt").read_text(encoding="utf-8").splitlines():
    declaration = raw.split(";", 1)[0].strip()
    if not declaration or declaration.startswith("#") or "==" not in declaration:
        continue
    package, wanted = (part.strip() for part in declaration.split("==", 1))
    normalized = re.sub(r"[-_.]+", "-", package).lower()
    actual = installed_distributions.get(normalized)
    if actual != wanted:
        pinned_failures.append(f"{package}: {actual or '未導入'}（期待値 {wanted}）")
if pinned_failures:
    raise RuntimeError("Forge固定依存の不一致:\n" + "\n".join(pinned_failures))
print("✅ Forge requirements.txtの固定依存を確認しました。")

current_forge_sha = git_head(FORGE_DIR)
if not USE_LATEST_FORGE_NEO and current_forge_sha != FORGE_NEO_SHA:
    raise RuntimeError(f"Forge Neoコミット不一致: {current_forge_sha}")
print(f"Forge Neo: {current_forge_sha}")
for name, spec in EXTENSIONS.items():
    if spec["enabled"]:
        actual = git_head(extensions_dir / name)
        print(f"{name}: {actual}")
        if actual != spec["sha"]:
            raise RuntimeError(f"拡張機能コミット不一致: {name}")

for name in MODEL_LINK_DIRS:
    link = local_models / name
    if not link.is_symlink() or link.resolve() != (MODEL_ROOT / name).resolve():
        raise RuntimeError(f"モデルリンクが不正です: {link}")
checkpoint_count = count_models(MODEL_ROOT / "Stable-diffusion")
lora_count = count_models(MODEL_ROOT / "Lora")
vae_files = sorted(p.name for p in (MODEL_ROOT / "VAE").rglob("*") if p.is_file() and p.suffix.lower() in MODEL_EXTENSIONS)
print(f"モデルルート: {MODEL_ROOT}")
print(f"Checkpoint: {checkpoint_count}件 / LoRA: {lora_count}件 / VAE: {len(vae_files)}件")
print("VAE一覧:", vae_files or "なし")
if checkpoint_count < 1:
    raise RuntimeError(f"Checkpointが見つかりません。{MODEL_ROOT / 'Stable-diffusion'} に配置してから再実行してください。")

output_resolved = OUTPUT_ROOT.resolve()
if output_resolved.is_relative_to(Path("/content/drive").resolve()):
    raise RuntimeError(f"出力先がGoogle Drive配下です: {output_resolved}")
if output_resolved != Path("/content/forge-neo-outputs"):
    raise RuntimeError(f"出力先が固定パスではありません: {output_resolved}")
help_text = run_checked([venv_python, FORGE_DIR / "launch.py", "--help"], cwd=FORGE_DIR, env=venv_env, capture=True, label="起動引数の確認").stdout
required_flags = ["--skip-prepare-environment", "--skip-version-check", "--share", "--gradio-auth-path"]
if ENABLE_API:
    required_flags += ["--api", "--api-auth"]
missing_flags = [flag for flag in required_flags if flag not in help_text]
if missing_flags:
    raise RuntimeError("固定コミットに必要な起動引数がありません: " + ", ".join(missing_flags))
print(f"出力先: {output_resolved}（Google Drive外）")
print("✅ 事前検証に合格しました。Web UIを起動できます。")

## 8. Forge Neo起動

認証付きGradio共有URLを発行します。このセルはWeb UIが終了するまで実行中になるのが正常です。実際の `https://....gradio.live` が表示された直下に、アクセス用ユーザー名とパスワードも続けて表示します。ログがない間も30秒ごとに生存状況を表示します。Forge標準の `launch.py` を直接起動し、スクリプト読込・UI callback・拡張情報読取りの例外をログから検出した場合は停止します。完全な起動ログは `/content/forge-neo-webui.log` に保存します。

パスワードは認証ファイルで渡すため起動コマンドに含めません。APIは既定OFFです。APIを有効にした場合だけ、同じ認証情報をAPI認証にも使用します。

**目的:** 認証付き共有URLでForge Neoを起動　**所要時間:** 初回5～15分、その後はWeb UI終了まで継続

In [ ]:
webui_password = secrets.token_urlsafe(24)
auth_path = CACHE_ROOT / "gradio-auth.txt"
auth_path.write_text(f"{WEBUI_USERNAME}:{webui_password}\n", encoding="utf-8")
os.chmod(auth_path, 0o600)
# Forge標準の起動経路を維持し、環境変数はPopenのenvで子プロセスへ直接渡す。
command = [venv_python, "-u", FORGE_DIR / "launch.py", "--skip-prepare-environment", "--skip-version-check", "--share", "--gradio-auth-path", auth_path]
if ENABLE_API:
    command += ["--api", "--api-auth", f"{WEBUI_USERNAME}:{webui_password}"]
masked_command = ["<認証情報>" if arg == f"{WEBUI_USERNAME}:{webui_password}" else str(arg) for arg in command]
print("=== Forge Neo 起動情報 ===")
print("ノートブック版:", NOTEBOOK_REVISION)
print("共有URLが発行された直後にアクセス用ユーザー名とパスワードを表示します。")
print("Forge Neo:", git_head(FORGE_DIR))
print("GPU:", [line for line in diagnostic.splitlines() if line.startswith("GPU:")][0])
print("モデル参照先:", MODEL_ROOT)
print("出力先:", OUTPUT_ROOT)
print("API:", "有効（認証あり）" if ENABLE_API else "無効")
print("完全ログ:", WEBUI_LOG_PATH)
print("起動コマンド:", " ".join(masked_command))
print("数分後、ログに表示される https://....gradio.live のURLを開いてください。")
launch_env = venv_env.copy()
launch_env.pop("UV_SYSTEM_PYTHON", None)
launch_env.pop("TRANSFORMERS_CACHE", None)
launch_env.update({"WEBUI_LAUNCH_LIVE_OUTPUT": "1", "PYTHONUNBUFFERED": "1", "UV_LINK_MODE": "hardlink", "MPLBACKEND": "Agg"})
backend_probe = run_checked([venv_python, "-c", "import os, matplotlib; print('MPLBACKEND=' + os.environ.get('MPLBACKEND', '<unset>')); print('matplotlib=' + str(matplotlib.get_backend()))"], env=launch_env, capture=True, label="起動直前Matplotlib診断").stdout.strip()
print(backend_probe)
if "MPLBACKEND=Agg" not in backend_probe or "matplotlib=Agg" not in backend_probe:
    raise RuntimeError("起動直前にMatplotlibをAggへ固定できませんでした。更新版ノートブックを先頭から再実行してください。")
try:
    run_checked(command, cwd=FORGE_DIR, env=launch_env, label="Forge Neo Web UI", startup_fatal_markers=("*** Error loading script:", "*** Error calling:", "*** Failed reading extension data from Git repository"), log_path=WEBUI_LOG_PATH, url_credentials=(WEBUI_USERNAME, webui_password))
except KeyboardInterrupt:
    print("Forge Neoを停止しました。生成画像はランタイム内に残っています。")

## 9. 生成画像のZIPダウンロード

出力全体の件数と合計サイズを確認し、固定パスのZIPを作ってブラウザへダウンロードします。空の場合は何もしません。

**目的:** ランタイム終了前に生成画像を取得　**所要時間:** 数秒～数分

In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile
from google.colab import files
output_files = [p for p in OUTPUT_ROOT.rglob("*") if p.is_file()]
total_size = sum(p.stat().st_size for p in output_files)
print(f"対象: {len(output_files)}ファイル / {total_size / 1024**2:.1f} MiB")
if not output_files:
    print("ダウンロード対象なし")
else:
    expected_zip = Path("/content/forge-neo-download.zip")
    if ZIP_PATH != expected_zip:
        raise RuntimeError(f"ZIP出力先が固定パスではありません: {ZIP_PATH}")
    ZIP_PATH.unlink(missing_ok=True)
    with ZipFile(ZIP_PATH, "w", compression=ZIP_DEFLATED, compresslevel=6) as archive:
        for path in output_files:
            archive.write(path, path.relative_to(OUTPUT_ROOT))
    print(f"ZIPを作成しました: {ZIP_PATH} ({ZIP_PATH.stat().st_size / 1024**2:.1f} MiB)")
    files.download(str(ZIP_PATH))

## 10. 任意：一時出力の消去

設定セルの `CONFIRM_DELETE_OUTPUTS = True` に変更した場合だけ、`/content/forge-neo-outputs` の中身を消去します。Google Driveは対象になりません。

**目的:** ランタイム内の生成画像だけを明示的に削除　**所要時間:** 数秒

In [ ]:
expected_output = Path("/content/forge-neo-outputs")
resolved_output = OUTPUT_ROOT.resolve()
delete_files = [p for p in OUTPUT_ROOT.rglob("*") if p.is_file()] if OUTPUT_ROOT.exists() else []
print(f"削除候補: {len(delete_files)}ファイル / 対象: {resolved_output}")
if not CONFIRM_DELETE_OUTPUTS:
    print("削除していません。実行する場合は設定セルでCONFIRM_DELETE_OUTPUTS=Trueにしてください。")
elif resolved_output != expected_output or resolved_output.is_relative_to(Path("/content/drive").resolve()):
    raise RuntimeError(f"安全確認に失敗したため削除しません: {resolved_output}")
else:
    for child in list(OUTPUT_ROOT.iterdir()):
        if child.is_dir() and not child.is_symlink():
            shutil.rmtree(child)
        else:
            child.unlink()
    for subdir in ["txt2img", "img2img", "extras", "grids", "saved", "temp"]:
        (OUTPUT_ROOT / subdir).mkdir(parents=True, exist_ok=True)
    print("✅ 一時出力だけを消去しました。Google Drive上のモデルは変更していません。")